# Day 4: Giving Agents Memory

An LLM on its own has **no memory**. Every call starts from a blank page and only knows what's inside the prompt. **Memory** is anything the agent carries forward to a later moment.

| Memory type | Simple meaning | Lasts |
|---|---|---|
| **Short-term** | The current conversation | Until the session ends |
| **Working** | The last few turns only (a sliding window) | Only the current step |
| **Semantic** | Facts that are true ("Priya is vegetarian") | Forever, and gets updated |
| **Episodic** | Events that happened ("Priya flagged INV-42") | Forever, and keeps growing |
| **Procedural** | Skills the agent knows how to run | Built by the developer |

We'll try each one on its own first, then combine all of them into one agent.

### Before you start

Create a file called `.env` in the same folder as this notebook and add your Azure OpenAI details:

```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_DEPLOYMENT=<your-chat-deployment-name>
AZURE_OPENAI_API_VERSION=2024-10-21
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=<your-embedding-deployment-name>
```

Never share this file or upload it to GitHub, because it contains your secret key.

### Setup
This cell loads your `.env` values and creates the `llm` object we use in the rest of this notebook. Run it first.

In [ ]:
# pip install -U langchain langgraph langchain-openai python-dotenv

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# 1. Load the values from the .env file
load_dotenv()

# 2. Make sure nothing important is missing
required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT", "AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError(f"Please add these to your .env file: {missing}")

# 3. Create the LLM
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    temperature=0,
)

# Embeddings turn text into numbers so we can search by meaning (used for episodic memory)
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)

print("Setup done. Using deployment:", os.environ["AZURE_OPENAI_DEPLOYMENT"])

In [ ]:
# Quick check: can we talk to the model?
print(llm.invoke("Say hello to the class in one short sentence.").content)

## 1. Short-term memory

`create_agent` builds a ready-made agent loop, and it runs on LangGraph under the hood. The **checkpointer** saves the conversation, and the **thread_id** acts like a chat window ID:

- **Same thread_id:** the agent remembers.
- **New thread_id:** it starts fresh, so the memory is "erased."

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(llm, tools=[], checkpointer=InMemorySaver())

def ask(agent, text, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
    return result["messages"][-1].content

ask(agent, "Hi, I'm Bob", thread_id="1")
print("Same thread:", ask(agent, "What's my name?", thread_id="1"))   # remembers Bob
print("New thread :", ask(agent, "What's my name?", thread_id="2"))   # forgot

## 2. Working memory

Working memory is a **sliding window**. We keep only the last few messages, and the oldest ones drop off first (first in, first out).

Here we keep the last 4 messages, so only turns 4 and 5 survive.

In [ ]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

chat = []
for n in range(1, 6):
    chat.append(HumanMessage(f"Turn {n} question"))
    chat.append(AIMessage(f"Turn {n} answer"))

window = trim_messages(chat, max_tokens=4, strategy="last",
                       token_counter=len)     # len = count messages, not tokens
print([m.content for m in window])

## 3. Long-term semantic memory, for facts

A **store** saves facts that survive across sessions. Each fact has:

- a **namespace**, like a folder: `("user_123", "facts")`
- a **key**, like a file name: `"city"`

Saving the same key again **replaces** the old value. This is called an **upsert**. It makes sense for facts, because a person has only one current city.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
FACTS = ("user_123", "facts")      # namespace = (user, type of memory)

store.put(FACTS, "city", {"value": "Hyderabad"})
store.put(FACTS, "city", {"value": "Bengaluru"})    # same key -> REPLACES old value
print(store.get(FACTS, "city").value)

store.delete(FACTS, "city")                          # clean up the demo

## 4. Episodic memory, for events

Episodes are **events**, like "the user flagged invoice INV-7." Unlike facts, we **append** every episode and never overwrite one.

To find episodes later, we search by **meaning**, not exact words. That's why the question "any billing problems?" finds the invoice episode, even though the words are different.

We also need a **write policy**. Not every message is worth remembering, so we only save the important ones.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

episodes = InMemoryVectorStore(embeddings)
episodes.add_texts(["User reported invoice INV-7 had the wrong amount."])
print(episodes.similarity_search("any billing problems?", k=1)[0].page_content)

episodes = InMemoryVectorStore(embeddings)           # reset for the lab


def worth_remembering(text):
    """Write policy: only save important events."""
    keywords = ["flag", "issue", "wrong", "complain", "refund", "problem"]
    return any(k in text.lower() for k in keywords)

print(worth_remembering("hello there"), worth_remembering("this invoice is wrong"))

## 5. Turn memory into tools, and add skills

The agent uses memory through **tools**, which are ordinary Python functions marked with `@tool`. The model reads the docstring to decide when to use each one.

**Procedural memory** is simply skills: tools the developer builds ahead of time. The agent doesn't work out *how* to approve an invoice. It just runs the skill.

In [ ]:
from langchain.tools import tool

# --- Semantic memory tools ---
@tool
def save_fact(key: str, value: str) -> str:
    """Save a fact or preference about the user. Example: key='diet', value='vegetarian'."""
    store.put(FACTS, key, {"value": value})
    return f"Saved: {key} = {value}"

@tool
def get_facts() -> str:
    """Get all saved facts about the user. Use before giving personal suggestions."""
    items = store.search(FACTS)
    return "\n".join(f"{i.key}: {i.value['value']}" for i in items) or "No facts yet."

# --- Episodic memory tool ---
@tool
def recall_past_events(query: str) -> str:
    """Search past conversations for events like earlier issues or complaints."""
    results = episodes.similarity_search(query, k=2)
    return "\n".join(doc.page_content for doc in results) or "Nothing found."

# --- Procedural memory: skills built ahead of time ---
@tool
def approve_invoice(invoice_id: str) -> str:
    """Approve an invoice by its ID, e.g. 'INV-42'."""
    return f"Invoice {invoice_id} approved."

@tool
def generate_report(table: str) -> str:
    """Generate a summary report for a data table, e.g. 'sales'."""
    return f"Report for {table}: 120 rows, total 4.2 lakh, top region South."

## 6. One agent with all memory types

| Memory | How it works in this agent |
|---|---|
| Short-term | The checkpointer (one thread per session) |
| Semantic | The `save_fact` and `get_facts` tools |
| Episodic | The `talk()` helper saves important messages; the `recall_past_events` tool finds them |
| Procedural | The `approve_invoice` and `generate_report` tools |

The **promotion rule** decides what moves from short-term into long-term memory. The model decides on facts, and our `worth_remembering` rule decides on events.

In [ ]:
memory_agent = create_agent(
    llm,
    tools=[save_fact, get_facts, recall_past_events, approve_invoice, generate_report],
    system_prompt="""You are an office assistant with memory.
- At the start of a conversation, call get_facts.
- If the user shares a preference or fact, call save_fact.
- If asked about something from before, call recall_past_events.""",
    checkpointer=InMemorySaver(),      # short-term memory
)

def talk(text, session):
    reply = ask(memory_agent, text, thread_id=session)
    if worth_remembering(text):                        # promote to episodic memory
        episodes.add_texts([f"[{session}] User said: {text}"])
    print(f"You  : {text}\nAgent: {reply}\n")

## 7. Hands-on lab: three sessions

Each session uses a **new thread_id**, so short-term memory is empty every time. If the agent still knows things in sessions 2 and 3, that knowledge came from **long-term** memory.

In [ ]:
print("===== Session 1 =====")
talk("Hi, I'm Priya. I'm vegetarian and I like short answers.", "session-1")
talk("Please flag invoice INV-42, the amount looks wrong.", "session-1")

print("===== Session 2 (new session) =====")
talk("What do you know about me?", "session-2")                # semantic
talk("Did I report any problem before?", "session-2")          # episodic
talk("It's fixed now. Approve INV-42.", "session-2")           # procedural

print("===== Session 3 =====")
talk("Generate a report for the sales table.", "session-3")    # procedural
talk("Suggest a dinner for me tonight.", "session-3")          # should be vegetarian

In [ ]:
# What ended up in long-term memory?
print("Saved facts   :", [(i.key, i.value["value"]) for i in store.search(FACTS)])
print("Saved episodes:", [d.page_content for d in episodes.similarity_search("invoice", k=5)])

## Key takeaways

| | Semantic | Episodic | Procedural |
|---|---|---|---|
| Stores | Facts (what's true) | Events (what happened) | Skills (what it can do) |
| Writing | Upsert: the new value replaces the old one | Append: every event is kept | Built by developers ahead of time |
| Reading | Exact lookup by key | Search by meaning | The LLM picks a tool |

- **Short-term memory** lasts one thread. **Long-term memory** lasts across sessions.
- **You decide** what's worth remembering. Saving everything just creates noise.
- `InMemoryStore` and `InMemorySaver` are lost when the notebook restarts. In production, you'd swap in a database (for example, `PostgresStore`).

**Try this:** In a new session, say "Actually I eat fish now." Then check that the `diet` fact was **replaced**, not duplicated.